# Laboratorio 2 - AlpesHearth

## Integrantes

- Isabella Naranjo
- David Caro

## Conjunto de datos resultante del laboratorio 1

El conjunto de datos del laboratorio 1 paso por un proceso de exploración 

In [ ]:
import pandas as pd 

df = pd.read_csv("../data/Datos Lab 1.csv")
df_model1 = df.copy()
df_model1 = df_model1.dropna(subset=["CVD Risk Score"])
before = len(df_model1)
df_model1 = df_model1.drop_duplicates(keep="first") #mantenemos solo la primera ocurrencia
after = len(df_model1)

print(f"Duplicados idénticos eliminados: {before - after}")

print("Antes:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
ids_duplicados = df_model1[df_model1["Patient ID"].duplicated(keep=False)]
df_model1 = df_model1[df_model1["CVD Risk Score"] >= 0]

cols_sin_target = df_model1.columns.tolist()
cols_sin_target.remove("CVD Risk Score")
cols_sin_target.remove("Patient ID")
df_base = df_model1.drop_duplicates(subset=["Patient ID"], keep="first")[["Patient ID"] + cols_sin_target]
df_score_prom = df_model1.groupby("Patient ID")["CVD Risk Score"].mean().reset_index()
df_model1 = df_base.merge(df_score_prom, on="Patient ID", how="left")

#verificacion
print("Después:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
print(df_model1.shape)

columns_to_drop = [
    'Patient ID',
    'Blood Pressure (mmHg)',
    'Height (cm)',
    'CVD Risk Level',
    'Date of Service'
]

columns_to_drop = [col for col in columns_to_drop if col in df_model1.columns]
df_model1 = df_model1.drop(columns=columns_to_drop)

df_model1 = df_model1[df_model1["Age"] >= 18]
df_model1 = df_model1[df_model1["Weight (kg)"] >= 30]
df_model1 = df_model1[df_model1["BMI"] >= 10]
df_model1 = df_model1[df_model1["Estimated LDL (mg/dL)"] >= 0]

print("Dimensiones después de eliminar valores imposibles:")
print(df_model1.shape)

numeric_cols = df_model1.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = [c for c in numeric_cols if c != "CVD Risk Score"]
print(numeric_cols)

def quitar_outliers(df, col):
    Q1 = df[col].quantile(0.25)   
    Q3 = df[col].quantile(0.75)   
    IQR = Q3 - Q1                 
    
    lower = Q1 - 1.5 * IQR        
    upper = Q3 + 1.5 * IQR        
    
    df_filtrado = df[(df[col] >= lower) & (df[col] <= upper)]
    
    print(f"{col}: antes={len(df)}, después={len(df_filtrado)}")
    return df_filtrado
for col in numeric_cols:
    df_model1 = quitar_outliers(df_model1, col)

print("Dimensiones finales después de quitar outliers:")
print(df_model1.shape)